# 中证800 V89：V46可解释风险否决层验证实验

目标不是重新排序，而是验证已公开暴露的异常风险能否减少 Top8 左尾损失。V46、raw-L2、fixed120、训练边界和板块约束保持不变；模型先产生 Top50，风险规则否决后由下一名补足 Top8。

五类规则均可独立开关：短期崩跌、持续回撤、交易异常、财务困境、三seed模型共识。输出平均收益、Top8 edge、最差月、月度CVaR10%、持仓底部2只均值、灾难持仓率、错误否决真实Top20比例及跨fold/seed稳定性。

风险行情数据优先读取训练CSV已有列；缺失时只为OOS候选股票调用聚宽行情API，并按月缓存。


## 0. 导入、目录与进度条


In [ ]:
import os
import gc
import warnings
import builtins as _bi
import datetime as _dt
from pathlib import Path

try:
    from jqdata import *
except Exception:
    pass

import lightgbm as lgb
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v89_risk_veto_outputs"
RISK_CACHE_DIR = PROJECT_DIR / "csi800_ml_v89_risk_cache"
FIGURE_DIR = OUT_DIR / "figures"
for _path in [OUT_DIR, RISK_CACHE_DIR, FIGURE_DIR]:
    _path.mkdir(parents=True, exist_ok=True)
RUN_TIMESTAMP = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
print("OUT_DIR:", OUT_DIR)


## 1. 冻结配置、规则开关与阈值


In [ ]:
DATA_PATH_OVERRIDE = None
DATA_CANDIDATES = [
    Path("v88_v46_universe_panel_20190101_20260713.csv"),
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
]
RUN_UNIVERSE = "csi800"

STOCK_COL = "stock"
DATE_COL = "rebalance_date"
NEXT_DATE_COL = "next_date"
FEATURE_DATE_COL = "feature_date"
TARGET_COL = "alpha_1m"
INDUSTRY_COL = "industry_bucket"

FOLD_SPECS = [
    {"fold_id": "oos_2022", "cutoff": "2021-12-31", "test_start": "2022-01-01", "test_end": "2022-12-31"},
    {"fold_id": "oos_2023", "cutoff": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"fold_id": "oos_2024", "cutoff": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"fold_id": "oos_2025", "cutoff": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"fold_id": "oos_2026", "cutoff": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]
SEEDS = [17, 42, 101]
FIXED_ITER = 120
CORR_THRESHOLD = 0.70
MIN_TRAIN_MONTHS = 30
NUM_THREADS = 4
CANDIDATE_POOL_SIZE = 50
CONSENSUS_TOP_N = 20
PRED_TOP_K = 8
TRUE_TOP_N = 20
BOARD_CAPS = {"chinext": 3, "star": 2}

RULE_SWITCHES = {
    "crash_5d": True,
    "drawdown_chain": True,
    "trading_anomaly": True,
    "financial_distress": True,
    "model_consensus": True,
}

# Frozen ex-ante thresholds. Change only in a new experiment version.
CRASH_RET5_MAX = -0.15
CRASH_Z_MAX = -3.0
DRAWDOWN_RET20_MAX = -0.25
DRAWDOWN_60_MAX = -0.30
LIMIT_DOWN_20_MAX = 2
PAUSED_20_MAX = 3
MIN_AVG_MONEY_20 = 50000000.0
HIGH_DEBT_PCT = 0.95
LOW_CASHFLOW_PCT = 0.10
HIGH_ACCA_PCT = 0.90
MIN_CONSENSUS_VOTES = 2
DISASTER_ALPHA_THRESHOLD = -0.20

BASE_PARAMS_FF10 = {
    "objective": "regression", "metric": "l2", "boosting_type": "gbdt",
    "learning_rate": 0.05, "num_leaves": 31, "min_data_in_leaf": 200,
    "feature_fraction": 1.0, "bagging_fraction": 0.8, "bagging_freq": 1,
    "lambda_l1": 0.1, "lambda_l2": 0.3, "verbose": -1,
}

SMOKE_TEST = False
if SMOKE_TEST:
    FOLD_SPECS = FOLD_SPECS[:1]
    SEEDS = [42]

print("RUN_UNIVERSE:", RUN_UNIVERSE)
print("enabled rules:", [k for k in RULE_SWITCHES if RULE_SWITCHES[k]])
print("models planned:", len(FOLD_SPECS) * len(SEEDS))


## 2. V46特征、实验变体与通用工具


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_V46_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)
RISK_DATA_COLS = [
    "risk_ret5", "risk_ret20", "risk_drawdown60", "risk_volatility20",
    "risk_money_mean20", "risk_paused_count20", "risk_limit_down_count20",
]
RISK_SOURCE_ALIASES = {
    "risk_ret5": ["risk_ret5", "px_ret_5"],
    "risk_ret20": ["risk_ret20", "px_ret_20"],
    "risk_drawdown60": ["risk_drawdown60", "px_drawdown_60"],
    "risk_volatility20": ["risk_volatility20", "px_volatility_20"],
    "risk_money_mean20": ["risk_money_mean20", "liq_money_mean_20", "audit_avg_money_20"],
    "risk_paused_count20": ["risk_paused_count20", "liq_paused_count_20"],
    "risk_limit_down_count20": ["risk_limit_down_count20", "liq_limit_down_count_20"],
}


def enabled(name):
    return bool(RULE_SWITCHES.get(name, False))


variant_specs = [{"variant": "baseline_no_gate", "rules": []}]
for name in ["crash_5d", "drawdown_chain", "trading_anomaly", "financial_distress", "model_consensus"]:
    if enabled(name):
        variant_specs.append({"variant": "only_" + name, "rules": [name]})
if enabled("crash_5d") and enabled("drawdown_chain"):
    variant_specs.append({"variant": "price_action_all", "rules": ["crash_5d", "drawdown_chain"]})
observable_rules = [x for x in ["crash_5d", "drawdown_chain", "trading_anomaly", "financial_distress"] if enabled(x)]
if len(observable_rules) >= 2:
    variant_specs.append({"variant": "observable_risk_all", "rules": observable_rules})
all_rules = [x for x in RULE_SWITCHES if enabled(x)]
if len(all_rules) >= 2:
    variant_specs.append({"variant": "all_gates", "rules": all_rules})

variant_manifest_df = pd.DataFrame([{
    "variant": x["variant"], "rules": ",".join(x["rules"]), "rule_count": len(x["rules"])
} for x in variant_specs])
variant_manifest_df.to_csv(OUT_DIR / "v89_rule_manifest.csv", index=False)
display_df(variant_manifest_df, 20)


def safe_mean(values):
    s = pd.Series(list(values), dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    return float(s.mean()) if len(s) else np.nan


def stock_board(stock):
    code_value = str(stock).split(".")[0]
    if code_value.startswith(("300", "301")):
        return "chinext"
    if code_value.startswith(("688", "689")):
        return "star"
    return "other"


def select_board_capped(ranked, rules):
    selected_indices = []
    board_counts = {"chinext": 0, "star": 0}
    for idx, row in ranked.iterrows():
        if _bi.any(bool(row.get("flag_" + rule, False)) for rule in rules):
            continue
        board = stock_board(row[STOCK_COL])
        if board in BOARD_CAPS and board_counts[board] >= BOARD_CAPS[board]:
            continue
        selected_indices.append(idx)
        if board in board_counts:
            board_counts[board] += 1
        if len(selected_indices) >= PRED_TOP_K:
            break
    return selected_indices


## 3. 数据加载、风险列映射与财务困境截面标记


In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        path = Path(DATA_PATH_OVERRIDE)
        if path.exists():
            return path
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % path)
    dynamic_v88 = _bi.sorted(PROJECT_DIR.glob("v88_v46_universe_panel_*.csv"), reverse=True)
    for item in dynamic_v88 + DATA_CANDIDATES:
        path = Path(item)
        if path.exists():
            return path
    raise IOError("training panel not found: %s" % [str(x) for x in DATA_CANDIDATES])


def load_dataset(path):
    header = pd.read_csv(path, nrows=0)
    required = [STOCK_COL, DATE_COL, NEXT_DATE_COL, TARGET_COL] + FULL_V46_COLS
    missing = [c for c in required if c not in header.columns]
    if missing:
        raise ValueError("dataset missing required columns: %s" % missing)
    optional = [FEATURE_DATE_COL, INDUSTRY_COL, "universe"]
    for aliases in RISK_SOURCE_ALIASES.values():
        optional.extend(aliases)
    usecols = unique_keep_order(required + [c for c in optional if c in header.columns])
    df = pd.read_csv(path, usecols=usecols)
    if "universe" in df.columns:
        df = df[df["universe"].astype(str) == RUN_UNIVERSE].copy()
    elif RUN_UNIVERSE != "csi800":
        raise ValueError("non-CSI800 run requires a V88 panel with universe column")
    df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce").dt.normalize()
    df[NEXT_DATE_COL] = pd.to_datetime(df[NEXT_DATE_COL], errors="coerce").dt.normalize()
    if FEATURE_DATE_COL in df.columns:
        df[FEATURE_DATE_COL] = pd.to_datetime(df[FEATURE_DATE_COL], errors="coerce").dt.normalize()
    else:
        df[FEATURE_DATE_COL] = df[DATE_COL] - pd.Timedelta(days=1)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    for target, aliases in RISK_SOURCE_ALIASES.items():
        source_col = _bi.next((c for c in aliases if c in df.columns), None)
        df[target] = pd.to_numeric(df[source_col], errors="coerce") if source_col else np.nan
    numeric_cols = unique_keep_order(FULL_V46_COLS + [TARGET_COL] + RISK_DATA_COLS)
    for col in progress_iter(numeric_cols, total=len(numeric_cols), desc="compact V89 data"):
        df[col] = pd.to_numeric(df[col], errors="coerce").astype(np.float32)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[STOCK_COL, DATE_COL, NEXT_DATE_COL, TARGET_COL])
    return df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)


DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)

# Financial flags use only same-date cross-sectional information and do not impute missing statements as distress.
debt_pct = df_all.groupby(DATE_COL)["debt_to_equity_ratio"].rank(method="average", pct=True)
cashflow_pct = df_all.groupby(DATE_COL)["net_operating_cash_flow_coverage"].rank(method="average", pct=True)
acca_pct = df_all.groupby(DATE_COL)["ACCA"].rank(method="average", pct=True)
df_all["flag_financial_distress"] = (
    (debt_pct >= HIGH_DEBT_PCT) & (cashflow_pct <= LOW_CASHFLOW_PCT) & (acca_pct >= HIGH_ACCA_PCT)
)
del debt_pct, cashflow_pct, acca_pct
gc.collect()

print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape, "months:", df_all[DATE_COL].nunique())
print("risk coverage:")
display_df(pd.DataFrame([{"risk_col": c, "coverage": float(df_all[c].notnull().mean())} for c in RISK_DATA_COLS]), 20)


## 4. V46 walk-forward训练并保留每月Top50候选


In [ ]:
def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            value = corr.iloc[i, j]
            if not pd.isnull(value) and abs(value) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    visited = set()
    components = []
    for col in feature_cols:
        if col in visited:
            continue
        stack = [col]
        component = []
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            component.append(current)
            stack.extend(graph[current])
        components.append(component)
    return components


def select_features_train_only(train_df):
    missing = train_df[FULL_V46_COLS].isnull().sum().to_dict()
    keep = []
    removed = []
    for component in build_corr_components(train_df, FULL_V46_COLS, CORR_THRESHOLD):
        ordered = _bi.sorted(component, key=lambda x: (missing[x], x))
        keep.append(ordered[0])
        removed.extend(ordered[1:])
    return keep, removed


def model_params(seed):
    params = dict(BASE_PARAMS_FF10)
    params.update({"seed": int(seed), "feature_fraction_seed": int(seed),
                   "bagging_seed": int(seed), "data_random_seed": int(seed), "num_threads": int(NUM_THREADS)})
    return params


candidate_rows = []
model_meta_rows = []
fold_plan_rows = []
candidate_meta_cols = unique_keep_order([
    STOCK_COL, DATE_COL, NEXT_DATE_COL, FEATURE_DATE_COL, TARGET_COL, INDUSTRY_COL,
    "flag_financial_distress"] + RISK_DATA_COLS)

for fold_spec in progress_iter(FOLD_SPECS, total=len(FOLD_SPECS), desc="V89 walk-forward folds"):
    cutoff = pd.Timestamp(fold_spec["cutoff"])
    train = df_all[(df_all[DATE_COL] <= cutoff) & (df_all[NEXT_DATE_COL] <= cutoff)].copy()
    test = df_all[(df_all[DATE_COL] >= pd.Timestamp(fold_spec["test_start"])) &
                  (df_all[DATE_COL] <= pd.Timestamp(fold_spec["test_end"]))].copy()
    train_months = int(train[DATE_COL].nunique())
    test_months = int(test[DATE_COL].nunique())
    status = "run" if train_months >= MIN_TRAIN_MONTHS and test_months > 0 else "skip"
    fold_plan_rows.append({"fold_id": fold_spec["fold_id"], "cutoff": fold_spec["cutoff"],
                           "train_rows": len(train), "train_months": train_months,
                           "test_rows": len(test), "test_months": test_months, "status": status})
    if status != "run":
        del train, test
        continue
    feature_cols, removed_cols = select_features_train_only(train)
    fill_values = train[feature_cols].median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X_train = train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32)
    y_train = train[TARGET_COL].astype(float)
    X_test = test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(fill_values).fillna(0).astype(np.float32).reset_index(drop=True)
    test_meta = test[candidate_meta_cols].copy().reset_index(drop=True)
    dtrain = lgb.Dataset(X_train, label=y_train, feature_name=list(feature_cols), free_raw_data=False)
    if hasattr(dtrain, "construct"):
        dtrain.construct()
    for seed in progress_iter(SEEDS, total=len(SEEDS), desc="V89 seeds %s" % fold_spec["fold_id"], leave=False):
        model = lgb.train(model_params(seed), dtrain, num_boost_round=int(FIXED_ITER))
        panel = test_meta.copy()
        panel["score"] = np.asarray(model.predict(X_test[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
        for date_value, month in progress_iter(panel.groupby(DATE_COL), total=panel[DATE_COL].nunique(),
                                                desc="collect Top50", leave=False):
            month = month.dropna(subset=[TARGET_COL, "score"]).copy()
            if len(month) < 30:
                continue
            month["true_rank"] = month[TARGET_COL].rank(method="first", ascending=False)
            month["universe_alpha"] = float(month[TARGET_COL].mean())
            top = month.sort_values("score", ascending=False).head(CANDIDATE_POOL_SIZE).copy()
            top["model_rank"] = np.arange(1, len(top) + 1)
            top["fold_id"] = fold_spec["fold_id"]
            top["seed"] = int(seed)
            candidate_rows.extend(top.to_dict("records"))
        model_meta_rows.append({"fold_id": fold_spec["fold_id"], "seed": int(seed),
                                "feature_count": len(feature_cols), "removed_features": ",".join(removed_cols),
                                "train_rows": len(train), "test_rows": len(test)})
        del model, panel
        gc.collect()
    del dtrain, X_train, X_test, y_train, test_meta, train, test
    gc.collect()

candidate_panel_df = pd.DataFrame(candidate_rows)
model_meta_df = pd.DataFrame(model_meta_rows)
fold_plan_df = pd.DataFrame(fold_plan_rows)
candidate_panel_df.to_csv(OUT_DIR / "v89_candidate_top50.csv", index=False)
model_meta_df.to_csv(OUT_DIR / "v89_model_meta.csv", index=False)
fold_plan_df.to_csv(OUT_DIR / "v89_fold_plan.csv", index=False)
print("candidate panel:", candidate_panel_df.shape)


## 5. 补齐候选风险行情并生成规则标记


In [ ]:
def fetch_risk_snapshot(stock_list, feature_date):
    try:
        price = get_price(stock_list, end_date=feature_date, frequency="daily",
                          fields=["close", "money", "paused", "low_limit"], count=61,
                          skip_paused=False, fq="pre", panel=False, fill_paused=True)
    except NameError:
        raise RuntimeError("risk columns are missing and JoinQuant get_price is unavailable")
    except Exception:
        price = None
    out = pd.DataFrame(index=stock_list, columns=RISK_DATA_COLS, dtype=float)
    if price is None or price.empty or "code" not in price.columns:
        return out
    price["time"] = pd.to_datetime(price["time"]).dt.normalize()
    close = price.pivot_table(index="time", columns="code", values="close").sort_index()
    money = price.pivot_table(index="time", columns="code", values="money").sort_index()
    paused = price.pivot_table(index="time", columns="code", values="paused").sort_index()
    low_limit = price.pivot_table(index="time", columns="code", values="low_limit").sort_index()
    returns = close.pct_change()
    last_close = close.iloc[-1]
    if len(close) >= 6:
        out["risk_ret5"] = last_close / close.iloc[-6] - 1
    if len(close) >= 21:
        out["risk_ret20"] = last_close / close.iloc[-21] - 1
    out["risk_drawdown60"] = last_close / close.tail(60).max() - 1
    out["risk_volatility20"] = returns.tail(20).std()
    out["risk_money_mean20"] = money.tail(20).mean()
    out["risk_paused_count20"] = paused.tail(20).fillna(0).sum()
    limit_down = (close.tail(20) <= low_limit.tail(20) * 1.001) & low_limit.tail(20).notnull()
    out["risk_limit_down_count20"] = limit_down.sum()
    return out.replace([np.inf, -np.inf], np.nan)


def complete_risk_data(candidate_df):
    outputs = []
    grouped = list(candidate_df.groupby(DATE_COL))
    for date_value, month in progress_iter(grouped, total=len(grouped), desc="complete candidate risk"):
        unique_meta = month.sort_values("seed").drop_duplicates(STOCK_COL, keep="last").copy()
        missing_any = unique_meta[RISK_DATA_COLS].isnull().any(axis=1)
        if missing_any.any():
            feature_date = pd.Timestamp(unique_meta[FEATURE_DATE_COL].dropna().iloc[-1])
            cache_path = RISK_CACHE_DIR / ("v89_risk_%s.csv" % pd.Timestamp(date_value).strftime("%Y%m%d"))
            cached = pd.read_csv(cache_path).set_index(STOCK_COL) if cache_path.exists() else pd.DataFrame()
            missing_stocks = [s for s in unique_meta.loc[missing_any, STOCK_COL].tolist() if s not in cached.index]
            if len(missing_stocks):
                fetched = fetch_risk_snapshot(missing_stocks, feature_date)
                fetched[STOCK_COL] = fetched.index
                cached = pd.concat([cached.reset_index() if len(cached) else pd.DataFrame(), fetched.reset_index(drop=True)], ignore_index=True)
                cached = cached.drop_duplicates(STOCK_COL, keep="last").set_index(STOCK_COL)
                cached.reset_index().to_csv(cache_path, index=False)
            for col in RISK_DATA_COLS:
                fill_map = cached[col] if col in cached.columns else pd.Series(dtype=float)
                unique_meta[col] = unique_meta[col].where(unique_meta[col].notnull(), unique_meta[STOCK_COL].map(fill_map))
        outputs.append(unique_meta)
    risk_meta = pd.concat(outputs, ignore_index=True)
    keep = [DATE_COL, STOCK_COL] + RISK_DATA_COLS
    merge_values = risk_meta[keep].drop_duplicates([DATE_COL, STOCK_COL], keep="last")
    out = candidate_df.drop(columns=RISK_DATA_COLS).merge(merge_values, on=[DATE_COL, STOCK_COL], how="left")
    return out


candidate_panel_df = complete_risk_data(candidate_panel_df)
vol_scale = candidate_panel_df["risk_volatility20"] * np.sqrt(5.0)
candidate_panel_df["risk_crash_z"] = candidate_panel_df["risk_ret5"] / vol_scale.where(vol_scale > 1e-8)
candidate_panel_df["flag_crash_5d"] = (
    (candidate_panel_df["risk_ret5"] <= CRASH_RET5_MAX)
    | (candidate_panel_df["risk_crash_z"] <= CRASH_Z_MAX)
).fillna(False)
candidate_panel_df["flag_drawdown_chain"] = (
    (candidate_panel_df["risk_ret20"] <= DRAWDOWN_RET20_MAX)
    & (candidate_panel_df["risk_drawdown60"] <= DRAWDOWN_60_MAX)
).fillna(False)
candidate_panel_df["flag_trading_anomaly"] = (
    (candidate_panel_df["risk_limit_down_count20"] >= LIMIT_DOWN_20_MAX)
    | (candidate_panel_df["risk_paused_count20"] >= PAUSED_20_MAX)
    | (candidate_panel_df["risk_money_mean20"] < MIN_AVG_MONEY_20)
).fillna(False)
candidate_panel_df.to_csv(OUT_DIR / "v89_candidate_top50_with_risk.csv", index=False)


## 6. 应用开关、补足Top8并审计误杀与左尾


In [ ]:
monthly_rows = []
selected_rows = []
veto_rows = []

date_groups = list(candidate_panel_df.groupby(["fold_id", DATE_COL]))
for keys, date_panel in progress_iter(date_groups, total=len(date_groups), desc="evaluate risk gates"):
    fold_id, date_value = keys
    vote_table = date_panel[date_panel["model_rank"] <= CONSENSUS_TOP_N].groupby(STOCK_COL)["seed"].nunique()
    date_panel = date_panel.copy()
    date_panel["consensus_votes"] = date_panel[STOCK_COL].map(vote_table).fillna(0).astype(int)
    date_panel["flag_model_consensus"] = date_panel["consensus_votes"] < MIN_CONSENSUS_VOTES
    for seed, seed_panel in date_panel.groupby("seed"):
        ranked = seed_panel.sort_values("model_rank").copy()
        baseline_indices = select_board_capped(ranked, [])
        baseline = ranked.loc[baseline_indices]
        baseline_set = set(baseline[STOCK_COL].tolist())
        for variant_spec in variant_specs:
            variant = variant_spec["variant"]
            rules = variant_spec["rules"]
            selected_indices = select_board_capped(ranked, rules)
            selected = ranked.loc[selected_indices].copy()
            selected_set = set(selected[STOCK_COL].tolist())
            removed = baseline[~baseline[STOCK_COL].isin(selected_set)].copy()
            added = selected[~selected[STOCK_COL].isin(baseline_set)].copy()
            selected_alpha = pd.to_numeric(selected[TARGET_COL], errors="coerce")
            bottom_n = _bi.min(2, len(selected_alpha))
            monthly_rows.append({
                "fold_id": fold_id, "rebalance_date": pd.Timestamp(date_value), "seed": int(seed),
                "variant": variant, "rules": ",".join(rules), "selected_count": int(len(selected)),
                "underfilled": bool(len(selected) < PRED_TOP_K),
                "replacement_count": int(len(removed)), "replacement_rate": len(removed) / float(PRED_TOP_K),
                "top8_alpha": float(selected_alpha.mean()) if len(selected_alpha) else np.nan,
                "top8_edge": float(selected_alpha.mean() - selected["universe_alpha"].iloc[0]) if len(selected_alpha) else np.nan,
                "precision_at8_true20": float((selected["true_rank"] <= TRUE_TOP_N).mean()) if len(selected) else np.nan,
                "bottom2_alpha_mean": float(selected_alpha.nsmallest(bottom_n).mean()) if bottom_n else np.nan,
                "disaster_holding_rate": float((selected_alpha <= DISASTER_ALPHA_THRESHOLD).mean()) if len(selected_alpha) else np.nan,
            })
            for _, row in selected.iterrows():
                selected_rows.append({
                    "fold_id": fold_id, "rebalance_date": pd.Timestamp(date_value), "seed": int(seed),
                    "variant": variant, "stock": row[STOCK_COL], "alpha_1m": row[TARGET_COL],
                    "true_rank": row["true_rank"], "model_rank": row["model_rank"],
                    "consensus_votes": row["consensus_votes"], "board": stock_board(row[STOCK_COL]),
                })
            removed_rows = removed.to_dict("records")
            added_rows = added.to_dict("records")
            pair_count = _bi.max(len(removed_rows), len(added_rows))
            for i in range(pair_count):
                old = removed_rows[i] if i < len(removed_rows) else {}
                new = added_rows[i] if i < len(added_rows) else {}
                old_alpha = old.get(TARGET_COL, np.nan)
                new_alpha = new.get(TARGET_COL, np.nan)
                triggered = [rule for rule in rules if bool(old.get("flag_" + rule, False))]
                veto_rows.append({
                    "fold_id": fold_id, "rebalance_date": pd.Timestamp(date_value), "seed": int(seed),
                    "variant": variant, "removed_stock": old.get(STOCK_COL, ""),
                    "removed_alpha": old_alpha, "removed_true_top20": bool(old.get("true_rank", 999999) <= TRUE_TOP_N),
                    "removed_disaster": bool(not pd.isnull(old_alpha) and old_alpha <= DISASTER_ALPHA_THRESHOLD),
                    "triggered_rules": ",".join(triggered), "added_stock": new.get(STOCK_COL, ""),
                    "added_alpha": new_alpha,
                    "replacement_gain": new_alpha - old_alpha if not pd.isnull(new_alpha) and not pd.isnull(old_alpha) else np.nan,
                })

monthly_metrics_df = pd.DataFrame(monthly_rows)
selected_detail_df = pd.DataFrame(selected_rows)
veto_detail_df = pd.DataFrame(veto_rows)
monthly_metrics_df.to_csv(OUT_DIR / "v89_rule_monthly.csv", index=False)
selected_detail_df.to_csv(OUT_DIR / "v89_selected_detail.csv", index=False)
veto_detail_df.to_csv(OUT_DIR / "v89_veto_replacement_detail.csv", index=False)
print("monthly:", monthly_metrics_df.shape, "veto pairs:", veto_detail_df.shape)


## 7. 跨seed/fold汇总、CVaR与决策表


In [ ]:
def cvar(values, q=0.10):
    s = pd.Series(values, dtype=float).replace([np.inf, -np.inf], np.nan).dropna().sort_values()
    if len(s) == 0:
        return np.nan
    n = _bi.max(1, int(np.ceil(len(s) * q)))
    return float(s.head(n).mean())


def summarize_variants(df, group_cols):
    rows = []
    grouped = list(df.groupby(group_cols))
    for keys, part in progress_iter(grouped, total=len(grouped), desc="summarize V89", leave=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict((group_cols[i], keys[i]) for i in range(len(group_cols)))
        row["months"] = int(len(part))
        for metric in ["top8_alpha", "top8_edge", "precision_at8_true20", "bottom2_alpha_mean",
                       "disaster_holding_rate", "replacement_rate", "underfilled"]:
            values = pd.to_numeric(part[metric], errors="coerce").dropna()
            row[metric + "_mean"] = float(values.mean()) if len(values) else np.nan
        row["top8_alpha_worst_month"] = float(part["top8_alpha"].min())
        row["top8_alpha_cvar10"] = cvar(part["top8_alpha"], 0.10)
        row["top8_edge_cvar10"] = cvar(part["top8_edge"], 0.10)
        rows.append(row)
    return pd.DataFrame(rows)


# Main summary averages seed portfolios by month before measuring time-series left tail.
seed_mean_monthly_df = monthly_metrics_df.groupby(["variant", DATE_COL])[
    ["top8_alpha", "top8_edge", "precision_at8_true20", "bottom2_alpha_mean",
     "disaster_holding_rate", "replacement_rate", "underfilled"]].mean().reset_index()
variant_summary_df = summarize_variants(seed_mean_monthly_df, ["variant"])
seed_summary_df = summarize_variants(monthly_metrics_df, ["variant", "seed"])
fold_summary_df = summarize_variants(monthly_metrics_df, ["variant", "fold_id"])

baseline = monthly_metrics_df[monthly_metrics_df["variant"] == "baseline_no_gate"].copy()
pair_keys = ["fold_id", DATE_COL, "seed"]
comparison_rows = []
for variant in [x["variant"] for x in variant_specs if x["variant"] != "baseline_no_gate"]:
    challenger = monthly_metrics_df[monthly_metrics_df["variant"] == variant].copy()
    pair = baseline[pair_keys + ["top8_alpha", "top8_edge", "bottom2_alpha_mean", "disaster_holding_rate"]].merge(
        challenger[pair_keys + ["top8_alpha", "top8_edge", "bottom2_alpha_mean", "disaster_holding_rate"]],
        on=pair_keys, suffixes=("_base", "_gate"))
    comparison_rows.append({
        "variant": variant, "paired_rows": int(len(pair)),
        "top8_alpha_delta_mean": float((pair["top8_alpha_gate"] - pair["top8_alpha_base"]).mean()),
        "top8_edge_delta_mean": float((pair["top8_edge_gate"] - pair["top8_edge_base"]).mean()),
        "bottom2_alpha_delta_mean": float((pair["bottom2_alpha_mean_gate"] - pair["bottom2_alpha_mean_base"]).mean()),
        "disaster_rate_delta_mean": float((pair["disaster_holding_rate_gate"] - pair["disaster_holding_rate_base"]).mean()),
        "positive_alpha_delta_rate": float(((pair["top8_alpha_gate"] - pair["top8_alpha_base"]) > 0).mean()),
    })
comparison_df = pd.DataFrame(comparison_rows)

veto_summary_rows = []
for variant, part in veto_detail_df.groupby("variant"):
    if variant == "baseline_no_gate":
        continue
    removed = part[part["removed_stock"].astype(str) != ""]
    veto_summary_rows.append({
        "variant": variant, "removed_count": int(len(removed)),
        "removed_alpha_mean": float(pd.to_numeric(removed["removed_alpha"], errors="coerce").mean()),
        "removed_disaster_rate": float(removed["removed_disaster"].mean()) if len(removed) else np.nan,
        "false_veto_true_top20_rate": float(removed["removed_true_top20"].mean()) if len(removed) else np.nan,
        "replacement_gain_mean": float(pd.to_numeric(part["replacement_gain"], errors="coerce").mean()),
    })
veto_summary_df = pd.DataFrame(veto_summary_rows)

decision = comparison_df.merge(variant_summary_df, on="variant", how="left").merge(veto_summary_df, on="variant", how="left")
decision["mean_alpha_not_damaged"] = decision["top8_alpha_delta_mean"] >= -0.001
base_summary = variant_summary_df[variant_summary_df["variant"] == "baseline_no_gate"].iloc[0]
decision["cvar10_improvement"] = decision["top8_alpha_cvar10"] - base_summary["top8_alpha_cvar10"]
decision["worst_month_improvement"] = decision["top8_alpha_worst_month"] - base_summary["top8_alpha_worst_month"]
decision["left_tail_improves"] = (decision["cvar10_improvement"] > 0) & (decision["bottom2_alpha_delta_mean"] > 0)
robust_rows = []
base_fold = fold_summary_df[fold_summary_df["variant"] == "baseline_no_gate"].set_index("fold_id")
base_seed = seed_summary_df[seed_summary_df["variant"] == "baseline_no_gate"].set_index("seed")
for variant in decision["variant"].tolist():
    fold_part = fold_summary_df[fold_summary_df["variant"] == variant].set_index("fold_id")
    seed_part = seed_summary_df[seed_summary_df["variant"] == variant].set_index("seed")
    common_folds = fold_part.index.intersection(base_fold.index)
    common_seeds = seed_part.index.intersection(base_seed.index)
    fold_cvar_wins = int((fold_part.loc[common_folds, "top8_alpha_cvar10"] > base_fold.loc[common_folds, "top8_alpha_cvar10"]).sum())
    seed_cvar_wins = int((seed_part.loc[common_seeds, "top8_alpha_cvar10"] > base_seed.loc[common_seeds, "top8_alpha_cvar10"]).sum())
    robust_rows.append({"variant": variant, "fold_cvar_wins": fold_cvar_wins,
                        "fold_total": len(common_folds), "seed_cvar_wins": seed_cvar_wins,
                        "seed_total": len(common_seeds)})
decision = decision.merge(pd.DataFrame(robust_rows), on="variant", how="left")
decision["robust_left_tail"] = (decision["fold_cvar_wins"] >= np.minimum(3, decision["fold_total"])) & (
    decision["seed_cvar_wins"] >= np.minimum(2, decision["seed_total"]))
decision["no_underfill"] = decision["underfilled_mean"] == 0
decision["decision"] = np.where(
    decision["mean_alpha_not_damaged"] & decision["left_tail_improves"] &
    decision["robust_left_tail"] & decision["no_underfill"],
    "risk_gate_candidate", "reject_or_observe")

outputs = {
    "v89_variant_summary.csv": variant_summary_df,
    "v89_seed_summary.csv": seed_summary_df,
    "v89_fold_summary.csv": fold_summary_df,
    "v89_pairwise_vs_baseline.csv": comparison_df,
    "v89_veto_effectiveness.csv": veto_summary_df,
    "v89_decision_table.csv": decision,
}
for filename, frame in progress_iter(list(outputs.items()), total=len(outputs), desc="save V89 summaries"):
    frame.to_csv(OUT_DIR / filename, index=False)
print("V89 decision")
display_df(decision.sort_values(["decision", "cvar10_improvement"], ascending=[True, False]), 30)


## 8. 可视化与输出说明


In [ ]:
def save_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.tight_layout()
    fig.savefig(str(path), dpi=140, bbox_inches="tight")
    try:
        plt.show()
    except Exception:
        pass
    plt.close(fig)
    print("saved figure:", path)


plot_df = decision.sort_values("top8_alpha_delta_mean")
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].barh(np.arange(len(plot_df)), plot_df["top8_alpha_delta_mean"].values, color="#2d8f78")
axes[0].axvline(0, color="#333333", linewidth=0.8)
axes[0].set_yticks(np.arange(len(plot_df)))
axes[0].set_yticklabels(plot_df["variant"], fontsize=9)
axes[0].set_title("Mean Top8 alpha delta vs baseline")
axes[1].barh(np.arange(len(plot_df)), plot_df["cvar10_improvement"].values, color="#d4543c")
axes[1].axvline(0, color="#333333", linewidth=0.8)
axes[1].set_yticks(np.arange(len(plot_df)))
axes[1].set_yticklabels(plot_df["variant"], fontsize=9)
axes[1].set_title("Monthly CVaR10 improvement")
save_figure(fig, "v89_return_and_left_tail_tradeoff.png")

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].barh(np.arange(len(plot_df)), plot_df["replacement_rate_mean"].values, color="#59636d")
axes[0].set_yticks(np.arange(len(plot_df)))
axes[0].set_yticklabels(plot_df["variant"], fontsize=9)
axes[0].set_title("Mean replacement rate")
axes[1].barh(np.arange(len(plot_df)), plot_df["false_veto_true_top20_rate"].values, color="#e39c24")
axes[1].set_yticks(np.arange(len(plot_df)))
axes[1].set_yticklabels(plot_df["variant"], fontsize=9)
axes[1].set_title("False veto: removed stock was true Top20")
save_figure(fig, "v89_veto_cost_and_false_positive.png")

readme_lines = [
    "V89 V46 explainable risk veto validation",
    "Run timestamp: %s" % RUN_TIMESTAMP,
    "Data: %s" % DATA_PATH,
    "Universe: %s" % RUN_UNIVERSE,
    "Rules are applied only after V46 Top50 scoring; replacements refill board-capped Top8.",
    "Primary result: v89_decision_table.csv",
    "Rule attribution: v89_veto_effectiveness.csv and v89_veto_replacement_detail.csv",
    "A valid gate must improve left tail without materially damaging mean alpha.",
    "This is an OOS ranking proxy experiment, not an event-driven execution backtest.",
]
with open(OUT_DIR / "v89_README.txt", "w") as f:
    f.write("\n".join(readme_lines))
print("outputs:")
for path in _bi.sorted(OUT_DIR.glob("*.csv")):
    print("-", path)
